## This script skletonizes all the masks of all the .tiff files in a folder and saves the xy coordinate of the skleton in results/results.csv file in the same directory as the tiff files.
The coordinate of the points in results csv are later used in ImportMasksToDLC.ipynb script to import in DLC and so on

In [1]:
# First cell: import necessary modules
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import pickle
import time

from backbone import Backbone
from candidate_point import CandidatePoints
from candidate_point_detect import CandidatePointsDetect
from constant import WORM, ROOT_SMOOTH
from graph import Graph
from graph_builder import GraphBuilder
from graph_prune import GraphPrune
from root_smooth import RootSmooth
from search_backbone import SearchBackbone
from skimage import io, color, morphology, measure
from scipy.ndimage import distance_transform_edt
from skimage import color, measure, morphology
from scipy.ndimage import distance_transform_edt
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
import pickle

In [2]:
# Second cell: utility functions
def load_skip_list(filename):
    skip_list = []
    try:
        with open(filename, 'r') as file:
            for line in file:
                skip_list.append(int(line.strip()))
    except IOError:
        pass
    return skip_list

def image_get(file_path, img_index):
    # First try the filename with zfill(5)
    image_filename_zfill = f"{file_path}/frame_{str(img_index).zfill(5)}.png"#f"{file_path}/{str(img_index).zfill(5)}.png"
    image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{img_index}.png"
    print(image_filename_zfill)
    if os.path.exists(image_filename_zfill):
        image_filename = image_filename_zfill
    elif not os.path.exists(image_filename):
        print(f"Pic {img_index} : Not Exist!")
        return None
    
    image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        print(f"Pic {img_index} : Failed to Load!")
        return None

    #gray = (image < 163) & (image > 161)
    #image = measure.label(gray).astype(np.uint8)
    
    if image.shape[0] != WORM.IMAGE_SIZE2 or image.shape[1] != WORM.IMAGE_SIZE1:
        print(f"Pic {img_index} : Wrong Image Size!")
        return None
    
    return image

def image_get2(file_path, img_index):
    image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{str(img_index).zfill(5)}.png"
    image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)
    gray =  (image < 179) & (image > 177)
    image = measure.label(gray).astype(np.uint8)
    
    if image is None or image.shape[0] != WORM.IMAGE_SIZE2 or image.shape[1] != WORM.IMAGE_SIZE1:
        print(f"Pic {img_index} : Not Exist or Wrong Image Size 1!")
        return None
    return image

def image_get1(file_path, img_index):
    image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{str(img_index).zfill(5)}.png"
    image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)
    
    if image is None:
        print(f"Pic {img_index} : Not Exist!")
        return None
    
    # Assuming we want to take the left half of the image
    left_half_image = image[:, :image.shape[1] // 2]
    
    gray = (left_half_image < 163) & (left_half_image > 161)
    labeled_image = measure.label(gray).astype(np.uint8)

    
    return labeled_image

def in_skip_list(index, skip_list):
    return index in skip_list


# Save results to CSV
def save_to_csv(save_path, image_path, ordered_points, max_thickness, length):
    frame_number = os.path.basename(image_path).split('.')[0].replace('frame', '')
    filename = os.path.join(save_path, "results.csv")
    ordered_points_flattened = [coord for point in ordered_points for coord in point]

    with open(filename, 'a') as file:
        file.write(','.join([frame_number] + [str(max_thickness)] + [str(length)] + list(map(str, ordered_points_flattened))) + '\n')



# Example gradient function
def interpolate_color(start_color, end_color, factor):
    return tuple([start_color[i] + factor * (end_color[i] - start_color[i]) for i in range(3)])

# Normalize the color values to 0-1 range for matplotlib
def normalize_color(color):
    return [c / 255.0 for c in color]

In [4]:
# define the main processing function
pic_start =0
pic_end = 11900#14400#12748

start_color = (255, 0, 0) # Red
end_color = (0, 0, 255)  # Blue


PIC_START = pic_start
PIC_END = pic_end


def process_folder(folder):
    search_backbone = SearchBackbone()
    A = folder#+'_masks'
    print(A)
    file_path = os.path.join(folder, A)
    save_folder = os.path.join(folder, 'results')
    save_folder_BBones = os.path.join(folder, 'backbones')
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)
    if not os.path.exists(save_folder_BBones):
        os.makedirs(save_folder_BBones)    

    for pic_num in range(PIC_START, PIC_END + 1):
        image = image_get(file_path, pic_num)
        if image is not None:
            filename = f"{save_folder_BBones}/backbone_{pic_num}.bin"

            try:
                start = time.time()
                backbone = search_backbone.search(image)
                end = time.time()
                print(f"Centerline extraction time consumption: {(end - start) * 1000}ms {pic_num}")

                distance_transform = distance_transform_edt(image)
                thicknesses = [distance_transform[int(y), int(x)] for y, x in search_backbone.temp_backbone.cood]
                max_thickness = max(thicknesses)
                length = search_backbone.temp_backbone.wormLength
                save_to_csv(save_folder, f"{file_path}/frame{pic_num}.tiff", search_backbone.temp_backbone.cood, max_thickness, length)

            except Exception as e:
                print(f"Error: {e}")
            search_backbone.save_centerline_results(filename)

In [5]:
pic_start =0
pic_end = 11900

start_color = (255, 0, 0)  # Red
end_color = (90, 0, 255)  # Blue


PIC_START = pic_start
PIC_END = pic_end


# List of folders to process
folders = [
    '/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all'  
]
for folder in folders:
    process_folder(folder)

/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00000.png
Error: '<' not supported between instances of 'NoneType' and 'int'
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00001.png
Normal
3
Error: index 3 is out of bounds for axis 0 with size 3
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00002.png
Error: 'NoneType' object has no attribute 'degree'
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00003.png
Error: '<' not supported between instances of 'NoneType' and 'int'
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00004.png
Normal
4
Error: index 4 is out of bounds for axis 0 with size 4
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00005.png
Normal
4
Error: index 4 is out of bounds for axis 0 with si

Omega
188
Current backbone length: 365.4947754108897, Mean length: 0.0
Centerline extraction time consumption: 290.32135009765625ms 91
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00092.png
Omega
194
Current backbone length: 356.0263348090406, Mean length: 365.4947754108897
Centerline extraction time consumption: 274.4615077972412ms 92
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00093.png
Normal
187
Current backbone length: 345.73139992187737, Mean length: 360.76055510996514
Centerline extraction time consumption: 265.0754451751709ms 93
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00094.png
Omega
170
Current backbone length: 328.9728680444441, Mean length: 355.75083671393594
Centerline extraction time consumption: 285.10236740112305ms 94
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00095.png
Omega
197
Current backbone length: 373.257653661

Normal
204
Current backbone length: 372.8652065944381, Mean length: 360.2111053263555
Centerline extraction time consumption: 295.67980766296387ms 124
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00125.png
Normal
197
Current backbone length: 365.3569136413031, Mean length: 360.6065459909831
Centerline extraction time consumption: 272.68362045288086ms 125
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00126.png
Normal
200
Current backbone length: 370.9391469603338, Mean length: 360.75049652584124
Centerline extraction time consumption: 283.34498405456543ms 126
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00127.png
Normal
203
Current backbone length: 374.0264622518338, Mean length: 361.05016271509106
Centerline extraction time consumption: 287.59765625ms 127
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00128.png
Normal
206
Current backbone leng

Normal
206
Current backbone length: 379.54310394942416, Mean length: 363.56243143285184
Centerline extraction time consumption: 308.300256729126ms 154
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00155.png
Normal
209
Current backbone length: 378.731224840531, Mean length: 363.86975205817055
Centerline extraction time consumption: 306.2772750854492ms 155
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00156.png
Normal
205
Current backbone length: 373.34921074560464, Mean length: 364.15015720500753
Centerline extraction time consumption: 293.5023307800293ms 156
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00157.png
Normal
211
Current backbone length: 368.49434414049085, Mean length: 364.32051004835193
Centerline extraction time consumption: 300.06980895996094ms 157
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00158.png
Normal
221
Current backbon

Omega
149
Current backbone length: 299.9750965701436, Mean length: 363.6338306046675
Centerline extraction time consumption: 507.0638656616211ms 184
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00185.png
Omega
153
Current backbone length: 297.30259686997323, Mean length: 363.6338306046675
Centerline extraction time consumption: 525.287389755249ms 185
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00186.png
Error: 'NoneType' object has no attribute 'degree'
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00187.png
Omega
187
Current backbone length: 371.3136585747103, Mean length: 363.6338306046675
Centerline extraction time consumption: 478.5888195037842ms 187
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00188.png
Omega
156
Current backbone length: 299.22901532305264, Mean length: 363.7435424328109
Centerline extraction time consumption: 562.2754

...

101
185
11
185
Omega
185
Current backbone length: 352.7780332228006, Mean length: 346.3921218737932
Centerline extraction time consumption: 366.3918972015381ms 11766
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11767.png
101
189
11
189
Omega
189
Current backbone length: 361.70968353399, Mean length: 346.39417720412763
Centerline extraction time consumption: 374.7670650482178ms 11767
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11768.png
101
199
11
199
Omega
199
Current backbone length: 354.90863232854, Mean length: 346.399104973217
Centerline extraction time consumption: 377.49695777893066ms 11768
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11769.png
101
191
11
191
Omega
191
Current backbone length: 358.4950931817839, Mean length: 346.4018420357308
Centerline extraction time consumption: 370.9900379180908ms 11769
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devid

101
198
11
198
Omega
198
Current backbone length: 362.6763518660889, Mean length: 346.51514307308344
Centerline extraction time consumption: 368.23058128356934ms 11798
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11799.png
101
200
11
200
Omega
200
Current backbone length: 365.23162455848, Mean length: 346.5202915945211
Centerline extraction time consumption: 357.5913906097412ms 11799
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11800.png
101
203
11
203
Omega
203
Current backbone length: 356.02821574876384, Mean length: 346.5262506177581
Centerline extraction time consumption: 353.40213775634766ms 11800
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11801.png
101
202
11
202
Omega
202
Current backbone length: 363.5737412516992, Mean length: 346.52927575788254
Centerline extraction time consumption: 361.38439178466797ms 11801
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavio

101
192
11
192
Omega
192
Current backbone length: 372.3542403219559, Mean length: 346.68200119035816
Centerline extraction time consumption: 360.0625991821289ms 11830
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11831.png
101
197
11
197
Omega
197
Current backbone length: 374.1835429108542, Mean length: 346.6900971345813
Centerline extraction time consumption: 362.93792724609375ms 11831
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11832.png
101
200
11
200
Omega
200
Current backbone length: 374.26213057785344, Mean length: 346.698764677386
Centerline extraction time consumption: 366.0919666290283ms 11832
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11833.png
101
203
11
203
Omega
203
Current backbone length: 360.8548121526179, Mean length: 346.70745152450246
Centerline extraction time consumption: 377.5913715362549ms 11833
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior

101
199
11
199
Omega
199
Current backbone length: 377.5736620750141, Mean length: 346.9088873205327
Centerline extraction time consumption: 349.7171401977539ms 11862
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11863.png
101
189
11
189
Omega
189
Current backbone length: 374.23431648572534, Mean length: 346.9184610872372
Centerline extraction time consumption: 348.4342098236084ms 11863
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11864.png
101
190
11
190
Omega
190
Current backbone length: 371.4658652175862, Mean length: 346.9269866351143
Centerline extraction time consumption: 360.6610298156738ms 11864
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11865.png
101
187
11
187
Omega
187
Current backbone length: 367.82877610650536, Mean length: 346.9346430714895
Centerline extraction time consumption: 373.39329719543457ms 11865
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior

101
220
11
220
Delta
220
Current backbone length: 411.93206909750234, Mean length: 347.0862958192549
Centerline extraction time consumption: 377.6519298553467ms 11893
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11894.png
101
202
11
202
Delta
202
Current backbone length: 404.45751651593866, Mean length: 347.0862958192549
Centerline extraction time consumption: 377.03967094421387ms 11894
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11895.png
101
204
11
204
Delta
204
Current backbone length: 381.74647233015486, Mean length: 347.0862958192549
Centerline extraction time consumption: 340.88730812072754ms 11895
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11896.png
Error: Prune Error!!! Special Node Num Must Be 2!!!
/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_11897.png
101
180
11
180
Delta
180
Current backbone length: 351.6817832405666, Mean len

In [6]:

image = image_get(folders[0], 0)


/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_00000.png


In [7]:
img_index = 10
file_path = folders[0]
image_filename = f"{file_path}/frame_{str(img_index)}_label.tiff"#f"{file_path}/{img_index}.png"
    
image = cv2.imread(image_filename, cv2.IMREAD_GRAYSCALE)

[ WARN:0@3567.314] global loadsave.cpp:248 findDecoder imread_('/mnt/DATA/Mahsa/movies/LongRecordings/2025_03_19/behavior_devided/Masks_all/frame_10_label.tiff'): can't open/read file: check file path/integrity
